# 🐞 Depuración en C — `gdb`, el equivalente de `pdb`

---

El tutorial principal de depuración usa **Python**, pero las mismas técnicas se aplican a C con herramientas equivalentes. Las dos diferencias clave son:

1. **Hay que compilar con `-g`** para incluir los símbolos de depuración en el ejecutable.
2. **El depurador se llama `gdb`** (GNU Debugger) en lugar de `pdb`.

Vamos a depurar exactamente el mismo bug (un incremento de más en un bucle `while`) pero ahora en C.

## El programa con bug en C

```c
/* Programa con un bug deliberado para practicar depuración. */
#include <stdio.h>

int main(void) {
    int n = 5;       /* esperamos: 2+4+6+8+10 = 30 */
    int suma = 0;
    int i = 0;

    while (i < n) {
        i = i + 1;
        suma = suma + 2 * i;
        i = i + 1;   /* 🐛 BUG: incremento de más */
    }

    printf("Suma de los %d primeros pares: %d\n", n, suma);
    printf("Esperado: 30\n");
    return 0;
}
```

Si lo compilas y ejecutas, obtienes:

```text
Suma de los 5 primeros pares: 18
Esperado: 30
```

¡Algo va mal! El bug es exactamente el mismo conceptual que en Python, da un resultado erróneo. Vamos a encontrarlo.

## 🐛 Técnica 1: `printf` debugging

Como en Python con `print`, en C basta con añadir `printf` estratégicos:

```c
while (i < n) {
    printf(">>> DEBUG: entrando con i=%d, suma=%d\n", i, suma);
    i = i + 1;
    suma = suma + 2 * i;
    printf(">>> DEBUG: tras la suma, i=%d, suma=%d\n", i, suma);
    i = i + 1;
    printf(">>> DEBUG: tras el segundo incremento, i=%d\n", i);
}
```

Recompilas con `gcc` y ejecutas. Verás los mismos saltos sospechosos que veíamos en Python: `i` avanza demasiado rápido.

## 🔬 Técnica 2: el depurador `gdb`

`gdb` (GNU Debugger) es el equivalente de `pdb` para C/C++. Para usarlo:

### Paso 1: Compilar con la opción `-g`

```bash
gcc -g -o programa_bug programa_con_bugs.c
```

La opción `-g` incluye en el ejecutable la información necesaria para que `gdb` sepa relacionar el código compilado con el código fuente.

### Paso 2: Arrancar `gdb`

```bash
gdb ./programa_bug
```

Se abre una consola interactiva `(gdb)` muy similar a la de `pdb`.

### Comandos esenciales de `gdb`

| Comando | Forma corta | Qué hace | Equivalente en `pdb` |
| :--- | :---: | :--- | :--- |
| `break N` | `b N` | Pone breakpoint en la línea N | (no hay sintaxis directa) |
| `run` | `r` | Inicia la ejecución | (no necesaria, pdb arranca al entrar) |
| `next` | `n` | Ejecuta la siguiente línea | `n` |
| `step` | `s` | Entra en una función llamada | `s` |
| `continue` | `c` | Continúa hasta el siguiente breakpoint | `c` |
| `print var` | `p var` | Imprime el valor de una variable | `p var` |
| `list` | `l` | Muestra el código fuente alrededor | `l` |
| `quit` | `q` | Sale | `q` |

### Ejemplo de sesión `gdb`

```text
$ gdb ./programa_bug
(gdb) break 11             # breakpoint en la línea del while
Breakpoint 1 at 0x...: file programa_con_bugs.c, line 11.
(gdb) run
Starting program: ./programa_bug

Breakpoint 1, main () at programa_con_bugs.c:11
11          while (i < n) {
(gdb) p i
$1 = 0
(gdb) p suma
$2 = 0
(gdb) n               # ejecutar la siguiente línea
12              i = i + 1;
(gdb) n
13              suma = suma + 2 * i;
(gdb) p i
$3 = 1                # i ha pasado a 1, correcto
(gdb) n
14              i = i + 1;     # ¿otra vez?
(gdb) p i
$4 = 1                # antes del segundo incremento
(gdb) n
(gdb) p i
$5 = 2                # 🚨 incremento sospechoso, este sobra
(gdb) quit
```

Igual que con `pdb`, ves rápidamente que **`i` avanza dos veces por cada vuelta**. Ahí está el bug.

## 💻 Técnica 3: VS Code con extensión de C

VS Code también permite depurar C de forma visual instalando la extensión **C/C++** (Microsoft). El flujo es exactamente el mismo que para Python:

1. Compilar con `gcc -g -o programa programa.c`.
2. Marcar breakpoint con click en el margen.
3. Pulsar **F5** para empezar la depuración.
4. Avanzar con los botones (Step Over, Step Into...) y ver las variables en el panel lateral.

## 🎯 Lo que debes recordar

* ✅ Compila siempre con **`-g`** cuando quieras depurar (sin esa opción, `gdb` no sabrá relacionar la línea ejecutada con el código fuente).
* ✅ Los comandos de `gdb` son **casi idénticos** a los de `pdb`: `next`, `print`, `continue`, `quit`...
* ✅ La **técnica** de depuración (poner breakpoints, inspeccionar variables, ejecutar paso a paso) es **la misma en todos los lenguajes**. Solo cambian las herramientas concretas.
* ✅ VS Code permite depurar C y Python con la misma interfaz visual, así que el esfuerzo de aprender el depurador es **rentable de por vida**.